# Veri Seti Kalitesi Analizi
Bu not defteri, `TUM_SONUCLAR_BIRLESIK_CLEAN.csv` veri setinin kalite durumunu değerlendirir.
Çalışma, veri dağılımı, eksik değer durumu, gürültü göstergeleri ve etiket güvenilirliğine odaklanır.

## 1. Veri Setini Yükleme
Veri setini pandas ile yükleyelim ve temel özelliklerine bakalım.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.35
})

csv_path = 'TUM_SONUCLAR_BIRLESIK_CLEAN.csv'
df = pd.read_csv(csv_path, low_memory=False)

print('Veri satır sayısı:', len(df))
print('Sütun sayısı:', len(df.columns))
print('Kolonlar:', df.columns.tolist())
print('\nÖrnek kayıtlar:')
display(df.head())

## 2. Temel İstatistiksel Özet
Sayısal ve kategorik sütunlar için tanımlayıcı istatistikleri hesaplayalım.

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print('Sayısal sütunlar:', numeric_cols)
print('Kategorik sütunlar:', categorical_cols)

print('Sayısal özet:')
display(df[numeric_cols].describe().T)

print('Kategorik özet:')
display(df[categorical_cols].nunique().sort_values(ascending=False).to_frame(name='unique_count'))

## 3. Veri Dağılımı Analizi
Faz, model, kalite bayrağı ve istasyon dağılımını inceleyelim.

In [ ]:
def plot_count(column, top_n=20, title=None):
    counts = df[column].value_counts(dropna=False).head(top_n)
    ax = counts.plot(kind='bar')
    ax.set_title(title or f'{column} dağılımı (ilk {top_n})')
    ax.set_ylabel('Sayı')
    ax.set_xlabel(column)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_count('phase', top_n=20, title='Faz dağılımı')
plot_count('model', top_n=25, title='Model dağılımı')
plot_count('pretrained', top_n=25, title='Önceden eğitilmiş model varyantları')
plot_count('quality_flag', top_n=25, title='Kalite bayrağı dağılımı')

top_stations = df['station'].value_counts().head(25)
ax = top_stations.plot(kind='bar')
ax.set_title('En çok kayıt içeren ilk 25 istasyon')
ax.set_ylabel('Kayıt sayısı')
ax.set_xlabel('Station')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df['confidence'].plot(kind='hist', bins=40, ax=ax[0], color='tab:blue')
ax[0].set_title('Confidence dağılımı')
ax[0].set_xlabel('Confidence')
ax[0].set_ylabel('Frekans')

df['duration_sec'].plot(kind='hist', bins=40, ax=ax[1], color='tab:orange')
ax[1].set_title('Duration (saniye) dağılımı')
ax[1].set_xlabel('Süre (saniye)')
ax[1].set_ylabel('Frekans')
plt.tight_layout()
plt.show()

## 4. Eksik ve Aykırı Değer Analizi
Eksik veri ve uç değerleri tespit ederek veri kalitesine etkisini analiz edelim.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Eksik değer sayıları:')
display(missing.to_frame(name='missing_count'))

duplicate_rows = df.duplicated().sum()
print(f'Tam kopya satır sayısı: {duplicate_rows}')

duration_stats = df['duration_sec'].describe()
print('Duration istatistikleri:')
print(duration_stats)

duration_neg = (df['duration_sec'] < 0).sum()
duration_zeros = (df['duration_sec'] == 0).sum()
print(f'Negatif duration sayısı: {duration_neg}')
print(f'Zero duration sayısı: {duration_zeros}')

duration_high = df[df['duration_sec'] > df['duration_sec'].quantile(0.995)]
print('En yüksek yüzde 0.5 lik duration değerlerinin örnek sayısı:', len(duration_high))
display(duration_high[['duration_sec', 'event_id', 'station', 'model', 'quality_flag']].head(10))

## 5. Gürültü Düzeyi Analizi
Mevcut meta veriye göre gürültü ve tutarsızlık göstergelerini inceleyelim.

In [ ]:
print('Kalite bayrağı kategorileri:')
display(df['quality_flag'].value_counts(dropna=False).to_frame(name='count'))

low_confidence = df[df['confidence'] < 0.5]
print('Confidence < 0.5 olan kayıt sayısı:', len(low_confidence))
display(low_confidence[['event_id', 'station', 'phase', 'model', 'confidence', 'quality_flag']].head(10))

flag_per_station = df.groupby('station')['quality_flag'].nunique().sort_values(ascending=False).head(20)
print('Birden fazla kalite bayrağına sahip ilk 20 istasyon:')
display(flag_per_station.to_frame('distinct_quality_flags'))

low_confidence_ratio = 100 * len(low_confidence) / len(df)
print(f'Düşük güven oranı: {low_confidence_ratio:.4f}%')

## 6. Etiket Güvenilirliği Değerlendirmesi
Aynı olay/istasyon/faz kombinasyonundaki çakışmaları ve tutarsızlıkları tespit edelim.

In [ ]:
group_cols = ['event_id', 'station', 'phase']
grouped = df.groupby(group_cols)
group_stats = grouped.agg(
    unique_models=('model', 'nunique'),
    unique_pick_times=('pick_time', 'nunique'),
    unique_quality_flags=('quality_flag', 'nunique'),
    min_confidence=('confidence', 'min'),
    max_confidence=('confidence', 'max'),
    count=('phase', 'count')
).reset_index()

multiple_labels = group_stats[(group_stats['unique_pick_times'] > 1) | (group_stats['unique_quality_flags'] > 1) | (group_stats['unique_models'] > 1)]
print('Çoklu etiket/tutarsız kayıt içeren grup sayısı:', len(multiple_labels))
print('Oran: %.4f%%' % (100 * len(multiple_labels) / len(group_stats)))
display(multiple_labels.sort_values(['count', 'unique_pick_times'], ascending=False).head(15))

same_trace = df.groupby(['trace_id_raw', 'phase']).agg(
    unique_events=('event_id', 'nunique'),
    unique_pick_times=('pick_time', 'nunique'),
    models=('model', lambda x: ','.join(sorted(set(x)))),
).reset_index()
conflict_trace = same_trace[(same_trace['unique_events'] > 1) | (same_trace['unique_pick_times'] > 1)]
print('Aynı trace_id/phase kombinasyonunda birden fazla olay veya pick_time olan kayıt sayısı:', len(conflict_trace))
display(conflict_trace.head(10))

## 7. Bilimsel Değerlendirme İçin Raporlama
Elde edilen bulguları özetleyelim ve veri altyapısının kalite değerlendirmesini sunalım.

In [ ]:
summary = {
    'Toplam kayıt': len(df),
    'Benzersiz event_id': df['event_id'].nunique(),
    'Benzersiz istasyon': df['station'].nunique(),
    'Faz sayısı': df['phase'].nunique(),
    'Model sayısı': df['model'].nunique(),
    'Kalite bayrağı sayısı': df['quality_flag'].nunique(),
    'Ortalama confidence': df['confidence'].mean(),
    'Confidence std': df['confidence'].std(),
    'Min confidence': df['confidence'].min(),
    'Max confidence': df['confidence'].max(),
    'Düşük confidence (<0.5) oranı (%)': 100 * len(df[df['confidence'] < 0.5]) / len(df),
    'Eksik değer var mı': df.isna().any().any(),
    'Tam kopya satır oranı (%)': 100 * duplicate_rows / len(df),
    'Çoklu etiket/tutarsız grup oranı (%)': 100 * len(multiple_labels) / len(group_stats),
}
summary_df = pd.DataFrame(list(summary.items()), columns=['Metirk', 'Değer'])
display(summary_df)

print('Not: Bu analiz sadece CSV içindeki meta veriye dayanır. Ham dalga formu verileri ve spektral gürültü ölçümleri mevcutsa daha derin bir gürültü değerlendirmesi yapılabilir.')